<a href="https://colab.research.google.com/github/BallFord/BigData26_A_2411531004_IqbalMiftahulFikri/blob/main/Praktikum%2002/BD_A_P02_2411531004_IqbalMiftahulFikri_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.chdir('/content/drive/MyDrive/BigData/Praktikum2')

print("Folder saat ini:", os.getcwd())

Folder saat ini: /content/drive/MyDrive/BigData/Praktikum2


# **K-1. Import Library dan Inisialisasi**

In [ ]:
!pip install faker

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

# **K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)**
### Kode ini mensimulasikan proses data acquisition dengan membuat 500 data transaksi marketplace sintetis yang memiliki berbagai masalah data mentah di dunia nyata, seperti format harga dan tanggal yang beraneka ragam, ketidakstabilan kapitalisasi huruf, missing value secara acak, serta duplikasi 15 baris transaksi (sehingga total menjadi 515 baris) sebelum akhirnya disimpan ke dalam file transaksi_mentah.csv.

In [ ]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


# **K-3. Deteksi dan Penanganan Missing Value**
### Kode df.isnull().sum() berfungsi untuk mendeteksi dan menghitung jumlah total nilai kosong pada setiap kolom di dalam DataFrame df. Langkah deteksi ini penting dilakukan di awal pra-pemrosesan data agar bisa melihat sebaran data yang hilang, sehingga dapat menentukan strategi penanganan yang tepat untuk masing-masing kolom, baik dengan menghapus barisnya maupun mengisinya dengan nilai pengganti.

In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


## **Strategi penanganan**
### Kode ini berfungsi untuk menangani missing value menggunakan dua pendekatan berbeda sesuai peran masing-masing kolom. Perintah dropna() menghapus baris yang kehilangan informasi vital seperti customer_name dan payment_method agar identitas transaksi tetap valid, sedangkan fillna() mengganti nilai kosong pada shipping_city dengan label "Tidak Diketahui" supaya sisa data transaksi pada baris tersebut tidak terbuang dan masih bisa dimanfaatkan untuk analisis.

In [ ]:
# Menghapus baris yang customer_name atau payment_method nya kosong
df = df.dropna(subset=["customer_name", "payment_method"])

# Mengisi nilai kosong pada shipping_city dengan "Tidak Diketahui"
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [ ]:
df.isnull().sum()

,0
transaction_id,0
customer_name,0
product_name,0
category,0
price,0
quantity,0
payment_method,0
transaction_date,0
shipping_city,0
rating,118


# **K-4. Deteksi dan Penanganan Duplicate**
### Kode di atas berfungsi untuk mendeteksi serta menghapus data transaksi ganda agar tidak terjadi penghitungan ganda pada tahap analisis berikutnya. Pertama, perintah df.duplicated().sum() menghitung jumlah baris yang seluruh nilainya persis sama serta duplikasi spesifik pada kolom transaction_id. Setelah itu, fungsi drop_duplicates() menghapus seluruh baris duplikat tersebut sehingga tersisa 500 baris data unik yang siap diproses lebih lanjut.

In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


# **K-5. Koreksi Tipe Data dan Standardisasi Format**

## **a. Standardisasi teks kategorikal (category, payment_method, shipping_city)**
### Kode ini berfungsi untuk menyamakan format penulisan pada kolom kategorikal (category, payment_method, dan shipping_city) agar tidak ada inkonsistensi akibat perbedaan huruf kapital maupun spasi liar. Method str.strip() menghapus spasi di awal dan akhir teks, str.title() mengubah format huruf menjadi Title Case, serta perintah .replace() mengembalikan kata "Cod" menjadi kapital penuh "COD" karena merupakan sebuah singkatan.

In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

## **b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)**
### Bagian ini bertujuan mengubah data pada kolom price yang semula berbentuk teks bercampur simbol menjadi nilai numerik sejati. Fungsi bersihkan_harga() akan mengikis semua karakter non angka tersebut dan mengonversi string bersih menjadi tipe float agar kolom harga bisa digunakan dalam perhitungan matematika atau analisis finansial.

In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

## **c. Standardisasi Format Tanggal ke YYYY-MM-DD**
### Kode ini menangani variasi format tanggal transaksi dengan mengubah semuanya ke satu format baku, yaitu YYYY-MM-DD. Penggunaan fungsi parse_tanggal() dengan mengecek format secara eksplisit satu per satu dilakukan untuk mencegah kesalahan pembacaan sebelum akhirnya dikonversi ke dalam string berformat seragam.

In [ ]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

## **d. Finalisasi Tipe Data**
### Tahap ini merupakan penegasan tipe data akhir untuk memastikan setiap kolom numerik tersimpan sesuai karakteristik datanya. Kolom quantity dikunci menggunakan tipe integer karena mewakili kuantitas barang yang selalu bernilai bulat, sedangkan kolom price dipastikan bertipe float untuk mengakomodasi nilai angka desimal.

In [ ]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [ ]:
# 1. Cek jumlah dan daftar nilai unik pada teks kategorikal
print("Jumlah kategori unik:", df['category'].nunique())
print("Daftar kategori:", df['category'].unique())
print("\nJumlah metode bayar unik:", df['payment_method'].nunique())
print("Daftar metode bayar:", df['payment_method'].unique())

# 2. Cek tipe data harga
print("\nTipe data kolom price:", df['price'].dtype)

# 3. Cek sampel format tanggal
print("\nSampel format tanggal:")
print(df['transaction_date'].head())

Jumlah kategori unik: 6
Daftar kategori: <StringArray>
['Rumah Tangga', 'Olahraga', 'Elektronik', 'Buku', 'Fashion', 'Kesehatan']
Length: 6, dtype: string

Jumlah metode bayar unik: 4
Daftar metode bayar: <StringArray>
['Transfer Bank', 'Kartu Kredit', 'E-Wallet', 'COD']
Length: 4, dtype: string

Tipe data kolom price: float64

Sampel format tanggal:
0    2026-09-10
1    2026-07-13
2    2026-09-05
3    2026-08-12
5    2026-08-10
Name: transaction_date, dtype: object


# **K-6. Ekspor Dataset Bersih**
### Kode ini berfungsi untuk mengekspor DataFrame df yang telah melalui seluruh proses pembersihan ke dalam file baru berformat CSV bernama transaksi_bersih.csv tanpa menyertakan indeks baris bawaan Pandas . Setelah file tersimpan, perintah print akan menampilkan konfirmasi jumlah akhir baris data bersih (sebanyak 490 baris) yang siap digunakan.

In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


# **Studi Kasus**

### 1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda lakukan.
Perbedaan angka tersebut terjadi karena angka 515 baris milik tim IT merupakan data mentah yang baru saja diambil dari sistem, sedangkan angka 490 baris milik tim Finance adalah data bersih yang telah melewati tahapan pembersihan.

Selisih 25 baris tersebut dihilangkan karena dua alasan utama:
- 15 baris merupakan data duplikat: Transaksi yang tercatat lebih dari satu kali akibat kegagalan koneksi atau glitch sistem saat pengiriman data.
- 10 baris mengalami kehilangan data wajib (missing value): Transaksi yang tidak memiliki nama pembeli (customer_name) atau metode pembayaran (payment_method), sehingga transaksi tersebut tidak memiliki identitas yang sah untuk diproses secara finansial.

### 2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity.
Ya, jumlah 490 baris jauh lebih benar dan dapat diandalkan dibandingkan 515 baris.

Hal ini berkaitan erat dengan dimensi Veracity dalam konsep 5V Big Data, yang mengukur seberapa akurat, valid, dan dapat dipercayanya suatu data untuk dijadikan dasar analisis atau pengambilan keputusan bisnis.

- Menggunakan 515 baris data mentah akan memicu prinsip "Garbage in, garbage out". Jika data duplikat dan data tanpa pembayaran ikut dihitung, pencatatan keuangan akan membengkak secara semu.
-	Dengan membuang 25 baris data bermasalah tersebut, nilai Veracity dari dataset meningkat pesat, sehingga laporan keuangan yang dihasilkan oleh tim Finance menjadi akurat, konsisten, dan mencerminkan kondisi bisnis yang sebenarnya.


### 3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?
Keputusan untuk membiarkan kolom rating tetap berisi nilai kosong (NaN) dapat dijelaskan kepada tim Finance sebagai berikut:

-	Sifat Data Opsional: Pemberian rating bersifat tidak wajib bagi pelanggan. Nilai kosong pada kolom ini mencerminkan perilaku asli pelanggan yang memilih untuk tidak memberikan ulasan, bukan disebabkan oleh kesalahan sistem.
-	Mencegah Distorsi Statistik: Jika kita memaksakan mengisi nilai kosong tersebut, misalnya mengisi dengan nilai rata-rata atau nilai netral, kita secara tidak langsung melakukan manipulasi data yang akan merusak keaslian tingkat kepuasan pelanggan yang sebenarnya.
-	Solusi Perhitungan Rata-Rata: Untuk mengetahui rating rata-rata semua transaksi, tim Finance cukup menghitung rata-rata dari transaksi yang benar-benar memiliki rating saja. Metode ini secara statistik sah dan merupakan standar yang benar untuk mengukur kepuasan pelanggan tanpa merusak integritas data.


# **Latihan**

## **Latihan 1: Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris transaksi_mentah.csv dan transaksi_bersih.csv dengan hasil SEED = 42. Apakah jumlahnya sama? Jelaskan mengapa.**

In [ ]:
import pandas as pd

df_mentah = pd.read_csv("transaksi_mentah.csv")
df_bersih = pd.read_csv("transaksi_bersih.csv")

print("Jumlah baris transaksi_mentah.csv :", len(df_mentah))
print("Jumlah baris transaksi_bersih.csv  :", len(df_bersih))

Jumlah baris transaksi_mentah.csv : 515
Jumlah baris transaksi_bersih.csv  : 490


In [ ]:
import pandas as pd

df_bersih = pd.read_csv("transaksi_bersih.csv")
print("Sampel 3 nama pelanggan pertama:")
print(df_bersih["customer_name"].head(3))

Sampel 3 nama pelanggan pertama:
0    Baktiadi Napitupulu, S.H.
1                Faizah Kusumo
2      Drs. Sari Aryani, M.TI.
Name: customer_name, dtype: object


### Jumlah baris pada transaksi_mentah.csv tetap sama persis (515 baris) baik pada SEED = 42 maupun SEED = 7 karena alur pembangkitan data sintetis menggunakan aturan eksplisit yang konstan, yaitu membangkitkan 500 data utama lalu menambahkan 15 baris duplikat. Sementara itu, pada transaksi_bersih.csv, jumlah akhirnya juga sama-sama menghasilkan 490 baris karena total baris cacat (missing value wajib dan duplikat) yang terbuang selama proses pembersihan secara statistik bernilai konstan (25 baris), meskipun entitas data dan sebaran nilai di dalamnya berubah 100% mengikuti seed generator angka acak yang baru. Dan yang berbeda itu hanya nilai pada rating nya saja (166 dan 120).

## **Latihan 2: Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk memeriksa apakah ada harga tidak valid.**

### Kode ini berfungsi untuk melakukan verifikasi kualitas data harga dengan menambahkan kolom boolean is_valid_price (price > 0), lalu menghitung sebarannya menggunakan value_counts() serta mengecek keberadaan nilai yang tidak valid (False). Hasil eksekusi menunjukkan seluruh 490 baris data bernilai True dan jumlah harga tidak valid bernilai 0, yang membuktikan bahwa proses pembersihan sebelumnya telah berhasil memastikan tidak ada harga bernilai 0, negatif, maupun anomali pada dataset.

In [ ]:
# Menambahkan kolom boolean is_valid_price
df["is_valid_price"] = df["price"] > 0

# Memeriksa jumlah data valid vs tidak valid
print("Hasil Validasi Kolom Price:")
print(df["is_valid_price"].value_counts())

# Memeriksa apakah ada harga yang tidak valid (False)
ada_tidak_valid = (df["is_valid_price"] == False).sum()
print(f"\nJumlah harga tidak valid: {ada_tidak_valid}")

Hasil Validasi Kolom Price:
is_valid_price
True    490
Name: count, dtype: int64

Jumlah harga tidak valid: 0


## **Latihan 3: Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang sudah bersih.**

### Kode ini berfungsi untuk menghitung frekuensi transaksi dari setiap kategori produk secara otomatis menggunakan fungsi value_counts() pada kolom category. Hasilnya kemudian ditampilkan secara terurut dari kategori terbanyak hingga tersedikit, yang menunjukkan bahwa seluruh 490 baris data bersih telah terkelompokkan dengan rapi ke dalam tiap kategori tanpa ada teks yang terduplikasi atau berantakan.

In [ ]:
# Menhitung jumlah transaksi untuk setiap kategori produk
jumlah_per_kategori = df["category"].value_counts()

print("Jumlah Transaksi per Kategori Produk:")
print(jumlah_per_kategori)

Jumlah Transaksi per Kategori Produk:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64
